In [3]:
import os
import glob
import xarray as xr
from IPython.display import HTML
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
from matplotlib.collections import QuadMesh
import cartopy.crs as ccrs
import cartopy.feature as cfeature

# Define directory and find the latest NetCDF file
nowcast_dir = '/dmidata/projects/energivejr/nowcasts'

'''
nc_files = glob.glob(os.path.join(nowcast_dir, '*.nc'))

if not nc_files:
    raise FileNotFoundError(f"No .nc files found in {nowcast_dir}")

latest_file = max(nc_files, key=os.path.getctime)
print(f"Loading latest nowcast: {latest_file}")
'''

# ...existing code...
# Choose file by timestamp substring present in filename, e.g. "SolarNowcast_202603170900.nc"
TIMESTAMP_SUBSTR = '202603170900'  # set to None to pick latest file

if TIMESTAMP_SUBSTR:
    # prefer the exact SolarNowcast_YYYYMMDDhhmm pattern, fallback to any file containing the substring
    pattern_primary = os.path.join(nowcast_dir, f"SolarNowcast_{TIMESTAMP_SUBSTR}*.nc")
    pattern_fallback = os.path.join(nowcast_dir, f"*{TIMESTAMP_SUBSTR}*.nc")
    matches = sorted(glob.glob(pattern_primary)) or sorted(glob.glob(pattern_fallback))
    if not matches:
        raise FileNotFoundError(f"No .nc files matching '{TIMESTAMP_SUBSTR}' in {nowcast_dir}")
    latest_file = max(matches, key=os.path.getctime)
else:
    nc_files = sorted(glob.glob(os.path.join(nowcast_dir, '*.nc')))
    if not nc_files:
        raise FileNotFoundError(f"No .nc files found in {nowcast_dir}")
    latest_file = max(nc_files, key=os.path.getctime)

print(f"Loading nowcast: {os.path.basename(latest_file)}")
# ...existing code...

# Load the dataset
ds = xr.open_dataset(latest_file)



# Infer data and time variable names if not set
time_var = 'time' if 'time' in ds.coords else list(ds.coords.keys())[0]
# pick first data variable (not coords)
data_vars = [v for v in ds.data_vars]
if not data_vars:
    raise RuntimeError("No data variables found in dataset `ds`.")
data_var = data_vars[0]

# Obtain lon/lat (try common names)
if 'lon' in ds.coords and 'lat' in ds.coords:
    lon = ds['lon'].values
    lat = ds['lat'].values
elif 'x' in ds.coords and 'y' in ds.coords:
    lon = ds['x'].values
    lat = ds['y'].values
else:
    # fallback: pick first two non-time coords
    coords = [k for k in ds.coords if k != time_var]
    lon = ds[coords[-1]].values
    lat = ds[coords[-2]].values

lon2d, lat2d = np.meshgrid(lon, lat)
extent = [float(lon.min()), float(lon.max()), float(lat.min()), float(lat.max())]

# Colormap and value range (use robust percentiles to avoid outliers)
cmap = 'gist_heat'  # nice perceptual cmap; change to 'viridis'/'cividis' if preferred
data_all = ds[data_var].values.astype(float)
vmin = float(np.nanpercentile(data_all, 2))
vmax = float(np.nanpercentile(data_all, 98))

# Set up figure & map
proj = ccrs.PlateCarree()
fig = plt.figure(figsize=(10, 8), facecolor='#0B1220')
ax = fig.add_subplot(1, 1, 1, projection=proj)
ax.set_facecolor('#0E1A2B')
ax.set_extent(extent, crs=proj)
ax.add_feature(cfeature.LAND.with_scale('50m'), facecolor='#0E1A2B', zorder=0)
ax.add_feature(cfeature.COASTLINE.with_scale('50m'), linewidth=0.6, edgecolor='white', zorder=2)
ax.add_feature(cfeature.BORDERS.with_scale('50m'), linewidth=0.4, edgecolor='white', zorder=2)
gl = ax.gridlines(draw_labels=True, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
gl.top_labels = False
gl.right_labels = False
gl.xlabel_style = gl.ylabel_style = {'size': 8, 'color': 'white'}

# Prepare first frame (squeeze leading singleton dims)
first_frame = np.squeeze(ds[data_var].isel({time_var: 0}).values)
if first_frame.ndim != 2:
    first_frame = first_frame.reshape(first_frame.shape[-2], first_frame.shape[-1])

im = ax.pcolormesh(lon2d, lat2d, first_frame,
                   transform=proj, cmap=cmap, vmin=vmin, vmax=vmax, shading='auto', rasterized=True)
cbar = fig.colorbar(im, ax=ax, orientation='vertical', fraction=0.046, pad=0.04)
cbar.set_label(data_var, color='white')
cbar.ax.yaxis.set_tick_params(color='white')
plt.setp(plt.getp(cbar.ax.axes, 'yticklabels'), color='white')

def animate(i):
    # remove previous QuadMesh to avoid accumulation
    for coll in list(ax.collections):
        if isinstance(coll, QuadMesh):
            coll.remove()

    data = np.squeeze(ds[data_var].isel({time_var: i}).values)
    if data.ndim != 2:
        data = data.reshape(data.shape[-2], data.shape[-1])

    pc = ax.pcolormesh(lon2d, lat2d, data,
                       transform=proj, cmap=cmap, vmin=vmin, vmax=vmax, shading='auto', rasterized=True)
    tstamp = ds[time_var].isel({time_var: i}).values
    ax.set_title(f"{data_var} — {np.datetime_as_string(tstamp, unit='s')}",
                 fontsize=12, color='white', pad=8)
    return [pc]

num_frames = ds.sizes[time_var]
ani = animation.FuncAnimation(fig, animate, frames=num_frames, interval=500, blit=False)

# Save GIF (requires pillow) — change filename or skip saving if not needed
output_gif = 'nowcast_map.gif'
ani.save(output_gif, writer='pillow', fps=5, dpi=300)
plt.close(fig)


Loading nowcast: SolarNowcast_202603170900.nc


In [5]:
# ...existing code...
import re
from matplotlib.collections import QuadMesh

# --- Create GIF from corresponding satellite images (NetCDF4_sds_YYYY-MM-DDThh_mm_ssZ.nc) ---
SAT_DIR = '/dmidata/projects/energivejr/satellite_data'

# Derive date string (YYYY-MM-DD) from TIMESTAMP_SUBSTR or from chosen nowcast filename
if TIMESTAMP_SUBSTR:
    date_ymd = TIMESTAMP_SUBSTR[:8]                      # 'YYYYMMDD'
    date_search = f"{date_ymd[:4]}-{date_ymd[4:6]}-{date_ymd[6:8]}"
else:
    m = re.search(r'(\d{4})(\d{2})(\d{2})', os.path.basename(latest_file))
    if not m:
        raise RuntimeError("Cannot infer date from nowcast filename; set TIMESTAMP_SUBSTR.")
    date_search = f"{m.group(1)}-{m.group(2)}-{m.group(3)}"

sat_pattern = os.path.join(SAT_DIR, f"NetCDF4_sds_{date_search}T*.nc")
sat_files = sorted(glob.glob(sat_pattern))
if not sat_files:
    raise FileNotFoundError(f"No satellite files found for {date_search} in {SAT_DIR}")

# Load all satellite frames (lazy load then compute arrays)
sat_dss = [xr.open_dataset(f) for f in sat_files]
# try common var name 'sds' (adjust if different)
if 'sds' not in sat_dss[0].data_vars:
    # pick first data var as fallback
    sat_var = list(sat_dss[0].data_vars)[0]
else:
    sat_var = 'sds'

# Get lon/lat from satellite file (try common names)
if 'lon' in sat_dss[0].coords and 'lat' in sat_dss[0].coords:
    sat_lon = sat_dss[0]['lon'].values
    sat_lat = sat_dss[0]['lat'].values
elif 'x' in sat_dss[0].coords and 'y' in sat_dss[0].coords:
    sat_lon = sat_dss[0]['x'].values
    sat_lat = sat_dss[0]['y'].values
else:
    coords = [k for k in sat_dss[0].coords]
    sat_lon = sat_dss[0][coords[-1]].values
    sat_lat = sat_dss[0][coords[-2]].values

sat_lon2d, sat_lat2d = np.meshgrid(sat_lon, sat_lat)
sat_extent = [float(sat_lon.min()), float(sat_lon.max()), float(sat_lat.min()), float(sat_lat.max())]

# Build stack of 2D arrays and compute robust vmin/vmax
sat_frames = []
for ds_sat in sat_dss:
    arr = ds_sat[sat_var].values.squeeze()
    if arr.ndim != 2:
        arr = arr.reshape(arr.shape[-2], arr.shape[-1])
    sat_frames.append(arr.astype(float))

stack = np.stack(sat_frames)
svmin = float(np.nanpercentile(stack, 2))
svmax = float(np.nanpercentile(stack, 98))

# Plot & animate (similar style as nowcast)
proj = ccrs.PlateCarree()
fig2 = plt.figure(figsize=(10, 8), facecolor='#0B1220')
ax2 = fig2.add_subplot(1, 1, 1, projection=proj)
ax2.set_facecolor('#0E1A2B')
ax2.set_extent(sat_extent, crs=proj)
ax2.add_feature(cfeature.LAND.with_scale('50m'), facecolor='#0E1A2B', zorder=0)
ax2.add_feature(cfeature.COASTLINE.with_scale('50m'), linewidth=0.6, edgecolor='white', zorder=2)
ax2.add_feature(cfeature.BORDERS.with_scale('50m'), linewidth=0.4, edgecolor='white', zorder=2)
gl2 = ax2.gridlines(draw_labels=True, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
gl2.top_labels = False
gl2.right_labels = False
gl2.xlabel_style = gl2.ylabel_style = {'size': 8, 'color': 'white'}

cmap_sat = 'gist_heat'  # perceptually-uniform, good for satellite irradiance
first_sat = stack[0]
im2 = ax2.pcolormesh(sat_lon2d, sat_lat2d, first_sat,
                     transform=proj, cmap=cmap_sat, vmin=svmin, vmax=svmax, shading='auto', rasterized=True)
cbar2 = fig2.colorbar(im2, ax=ax2, orientation='vertical', fraction=0.046, pad=0.04)
cbar2.set_label(sat_var, color='white')
cbar2.ax.yaxis.set_tick_params(color='white')
plt.setp(plt.getp(cbar2.ax.axes, 'yticklabels'), color='white')

def animate_sat(i):
    # remove previous QuadMesh objects
    for coll in list(ax2.collections):
        if isinstance(coll, QuadMesh):
            coll.remove()

    data = stack[i]
    pc = ax2.pcolormesh(sat_lon2d, sat_lat2d, data,
                        transform=proj, cmap=cmap_sat, vmin=svmin, vmax=svmax, shading='auto', rasterized=True)
    # timestamp from filename
    tstamp = os.path.basename(sat_files[i]).replace('.nc', '').split('_')[-1]
    ax2.set_title(f"Satellite {sat_var} — {tstamp}", fontsize=12, color='white', pad=8)
    return [pc]

sat_num_frames = len(stack)
ani_sat = animation.FuncAnimation(fig2, animate_sat, frames=sat_num_frames, interval=500, blit=False)

output_sat_gif = f"satellite_{date_search}.gif"
ani_sat.save(output_sat_gif, writer='pillow', fps=5, dpi=300)
plt.close(fig2)

# close opened datasets
for d in sat_dss:
    d.close()

# display
HTML(f'<p>Satellite GIF: {output_sat_gif}</p><img src="{output_sat_gif}">')
# ...existing code...

In [6]:
# ...existing code...
import re
from matplotlib.collections import QuadMesh

# --- Create GIF from corresponding satellite images (NetCDF4_sds_YYYY-MM-DDThh_mm_ssZ.nc) ---
SAT_DIR = '/dmidata/projects/energivejr/satellite_data'

# Derive date string (YYYY-MM-DD) from TIMESTAMP_SUBSTR or from chosen nowcast filename
if TIMESTAMP_SUBSTR:
    date_ymd = TIMESTAMP_SUBSTR[:8]                      # 'YYYYMMDD'
    date_search = f"{date_ymd[:4]}-{date_ymd[4:6]}-{date_ymd[6:8]}"
else:
    m = re.search(r'(\d{4})(\d{2})(\d{2})', os.path.basename(latest_file))
    if not m:
        raise RuntimeError("Cannot infer date from nowcast filename; set TIMESTAMP_SUBSTR.")
    date_search = f"{m.group(1)}-{m.group(2)}-{m.group(3)}"

sat_pattern = os.path.join(SAT_DIR, f"NetCDF4_sds_{date_search}T*.nc")
sat_files = sorted(glob.glob(sat_pattern))
if not sat_files:
    raise FileNotFoundError(f"No satellite files found for {date_search} in {SAT_DIR}")

# parse satellite file timestamps (filename contains 'YYYY-MM-DDThh_mm_ssZ')
sat_times = []
ts_re = re.compile(r'(\d{4}-\d{2}-\d{2}T\d{2}_\d{2}_\d{2}Z)')
for f in sat_files:
    m = ts_re.search(os.path.basename(f))
    if m:
        tstr = m.group(1).rstrip('Z').replace('_', ':')  # '2026-03-17T13:45:00'
        sat_times.append(np.datetime64(tstr))
    else:
        # fallback to file modification time if parsing fails
        sat_times.append(np.datetime64(int(os.path.getmtime(f)), 's'))

# determine nowcast start time (use first frame or chosen START_IDX if present)
nowcast_start = np.squeeze(ds[time_var].isel({time_var: 0}).values)
nowcast_start = np.datetime64(nowcast_start)

# find satellite index that best matches (first sat >= nowcast_start, else nearest)
start_idx = None
for i, st in enumerate(sat_times):
    if st >= nowcast_start:
        start_idx = i
        break
if start_idx is None:
    # all sat_times < nowcast_start -> pick closest
    start_idx = int(np.argmin([abs((st - nowcast_start).astype('timedelta64[s]').astype(int)) for st in sat_times]))

# decide how many frames to animate (match nowcast frames if possible)
nowcast_nframes = ds.sizes[time_var]
available = len(sat_files) - start_idx
sat_nframes = min(available, nowcast_nframes)

# select satellite files and times to use
sat_files_sel = sat_files[start_idx : start_idx + sat_nframes]
sat_times_sel = sat_times[start_idx : start_idx + sat_nframes]

# Load the selected satellite frames
sat_dss = [xr.open_dataset(f) for f in sat_files_sel]
# try common var name 'sds' (adjust if different)
if 'sds' not in sat_dss[0].data_vars:
    sat_var = list(sat_dss[0].data_vars)[0]
else:
    sat_var = 'sds'

# get lon/lat
if 'lon' in sat_dss[0].coords and 'lat' in sat_dss[0].coords:
    sat_lon = sat_dss[0]['lon'].values
    sat_lat = sat_dss[0]['lat'].values
elif 'x' in sat_dss[0].coords and 'y' in sat_dss[0].coords:
    sat_lon = sat_dss[0]['x'].values
    sat_lat = sat_dss[0]['y'].values
else:
    coords = [k for k in sat_dss[0].coords]
    sat_lon = sat_dss[0][coords[-1]].values
    sat_lat = sat_dss[0][coords[-2]].values

sat_lon2d, sat_lat2d = np.meshgrid(sat_lon, sat_lat)
sat_extent = [float(sat_lon.min()), float(sat_lon.max()), float(sat_lat.min()), float(sat_lat.max())]

# Build stack and vmin/vmax
sat_frames = []
for ds_sat in sat_dss:
    arr = ds_sat[sat_var].values.squeeze()
    if arr.ndim != 2:
        arr = arr.reshape(arr.shape[-2], arr.shape[-1])
    sat_frames.append(arr.astype(float))
stack = np.stack(sat_frames)
svmin = float(np.nanpercentile(stack, 2))
svmax = float(np.nanpercentile(stack, 98))

# Plot & animate (aligned start)
proj = ccrs.PlateCarree()
fig2 = plt.figure(figsize=(10, 8), facecolor='#0B1220')
ax2 = fig2.add_subplot(1, 1, 1, projection=proj)
ax2.set_facecolor('#0E1A2B')
ax2.set_extent(sat_extent, crs=proj)
ax2.add_feature(cfeature.LAND.with_scale('50m'), facecolor='#0E1A2B', zorder=0)
ax2.add_feature(cfeature.COASTLINE.with_scale('50m'), linewidth=0.6, edgecolor='white', zorder=2)
ax2.add_feature(cfeature.BORDERS.with_scale('50m'), linewidth=0.4, edgecolor='white', zorder=2)
gl2 = ax2.gridlines(draw_labels=True, linewidth=0.3, color='gray', alpha=0.4, linestyle='--')
gl2.top_labels = False
gl2.right_labels = False
gl2.xlabel_style = gl2.ylabel_style = {'size': 8, 'color': 'white'}

cmap_sat = 'gist_heat'
first_sat = stack[0]
im2 = ax2.pcolormesh(sat_lon2d, sat_lat2d, first_sat,
                     transform=proj, cmap=cmap_sat, vmin=svmin, vmax=svmax, shading='auto', rasterized=True)
cbar2 = fig2.colorbar(im2, ax=ax2, orientation='vertical', fraction=0.046, pad=0.04)
cbar2.set_label(sat_var, color='white')
cbar2.ax.yaxis.set_tick_params(color='white')
plt.setp(plt.getp(cbar2.ax.axes, 'yticklabels'), color='white')

def animate_sat(i):
    for coll in list(ax2.collections):
        if isinstance(coll, QuadMesh):
            coll.remove()
    data = stack[i]
    pc = ax2.pcolormesh(sat_lon2d, sat_lat2d, data,
                        transform=proj, cmap=cmap_sat, vmin=svmin, vmax=svmax, shading='auto', rasterized=True)
    tstamp = sat_times_sel[i]
    ax2.set_title(f"Satellite {sat_var} — {np.datetime_as_string(tstamp, unit='s')}", fontsize=12, color='white', pad=8)
    return [pc]

sat_num_frames = stack.shape[0]
ani_sat = animation.FuncAnimation(fig2, animate_sat, frames=sat_num_frames, interval=500, blit=False)

output_sat_gif = f"satellite_map_aligned.gif"
ani_sat.save(output_sat_gif, writer='pillow', fps=5, dpi=300)
plt.close(fig2)

# close opened datasets
for d in sat_dss:
    d.close()

# display
HTML(f'<p>Satellite GIF (aligned start): {output_sat_gif}</p><img src="{output_sat_gif}">')
# ...existing code...